### Testing reddit

In [45]:
import os
import os
import sys
import requests
import openai
import json
from pprint import pprint

from dotenv import load_dotenv

dotenv_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path)

openai_api_key = os.getenv('OPENAI_API_KEY')


## Reddit search function

In [87]:
def reddit_search(input_text, client_id, client_secret, user_agent, subreddit=None, time_filter=None, sort=None, limit=10) -> list:
    """
    Executes a Reddit search based on the input query and parameters using Reddit's API directly.
    """
    auth = requests.auth.HTTPBasicAuth(client_id, client_secret)
    data = {
        'grant_type': 'client_credentials'
    }
    headers = {'User-Agent': user_agent}

    # Get OAuth token
    res = requests.post('https://www.reddit.com/api/v1/access_token', auth=auth, data=data, headers=headers)
    token = res.json()['access_token']

    headers['Authorization'] = f'bearer {token}'

    params = {
        'q': input_text,
        'limit': limit,
        'sort': sort or 'relevance',
        't': time_filter or 'all',
        'restrict_sr': True if subreddit else False,
    }

    url = f"https://oauth.reddit.com/r/{subreddit}/search" if subreddit else "https://oauth.reddit.com/search"
    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        raise Exception(f"Reddit API call failed: {response.text}")
    
    json_data = response.json()
    results:list = [child['data'] for child in json_data.get('data', {}).get('children', [])]

    return results

### Test

In [88]:
INPUT_TEXT = "Most popular commander in Magic the Gathering"
CLIENT_ID = os.getenv('REDDIT_CLIENT_ID')
CLIENT_SECRET = os.getenv('REDDIT_CLIENT_SECRET')
USER_AGENT = os.getenv('REDDIT_USER_AGENT')

In [89]:
search_res = reddit_search(input_text = INPUT_TEXT, 
              client_id = CLIENT_ID, 
              client_secret = CLIENT_SECRET, 
              user_agent = USER_AGENT, 
              limit = 2)

print(search_res)

[{'approved_at_utc': None, 'subreddit': 'HobbyDrama', 'selftext': '**Background**\n\nFor all the Magic the Gathering posts on this subreddit, it’s shocking to me that one man has not been brought up: DesolatorMagic. Much has been written about his once friend [Jeremy Hambly](https://www.reddit.com/r/HobbyDrama/comments/sw9q7l/magic_the_gathering_the_downfall_of/?utm_source=share&amp;utm_medium=ios_app&amp;utm_name=iossmf) seen [here](https://archive.fo/YL3yz) hanging out together in of an anime convention that Desolator was banned from (more on that later), nothing has been written about perhaps the most infamous whiner in magic history. I’m here to fix that. Strap in, because it’s going to get *wild*. Let’s begin. \n\n\n**Humble Beginnings**\n\nThough Des’ internet life extends farther than his Magic the Gathering notoriety, that’s a post for another time. He started posting on various message boards under the pseudonym “Desolator114” around 2006, and made his YouTube channel the same

In [95]:
first_res = search_res[0] if search_res else None

if first_res:
	print(first_res.keys())
	print(f"\nThere are {len(search_res)} items in the search result")
	print("looking at first item in the search result...")
	# Uncomment the following line if you want to inspect the keys of the first result
	print(first_res.keys())
else:
	print("No search results found.")

dict_keys(['approved_at_utc', 'subreddit', 'selftext', 'author_fullname', 'saved', 'mod_reason_title', 'gilded', 'clicked', 'title', 'link_flair_richtext', 'subreddit_name_prefixed', 'hidden', 'pwls', 'link_flair_css_class', 'downs', 'thumbnail_height', 'top_awarded_type', 'hide_score', 'name', 'quarantine', 'link_flair_text_color', 'upvote_ratio', 'author_flair_background_color', 'subreddit_type', 'ups', 'total_awards_received', 'media_embed', 'thumbnail_width', 'author_flair_template_id', 'is_original_content', 'user_reports', 'secure_media', 'is_reddit_media_domain', 'is_meta', 'category', 'secure_media_embed', 'link_flair_text', 'can_mod_post', 'score', 'approved_by', 'is_created_from_ads_ui', 'author_premium', 'thumbnail', 'edited', 'author_flair_css_class', 'author_flair_richtext', 'gildings', 'content_categories', 'is_self', 'mod_note', 'created', 'link_flair_type', 'wls', 'removed_by_category', 'banned_by', 'author_flair_type', 'domain', 'allow_live_comments', 'selftext_html', 

### Go through keys

In [98]:
for k in first_res.keys():
    if k is not None:
        print(f"{k}: {first_res.get(k)}")


approved_at_utc: None
subreddit: HobbyDrama
selftext: **Background**

For all the Magic the Gathering posts on this subreddit, it’s shocking to me that one man has not been brought up: DesolatorMagic. Much has been written about his once friend [Jeremy Hambly](https://www.reddit.com/r/HobbyDrama/comments/sw9q7l/magic_the_gathering_the_downfall_of/?utm_source=share&amp;utm_medium=ios_app&amp;utm_name=iossmf) seen [here](https://archive.fo/YL3yz) hanging out together in of an anime convention that Desolator was banned from (more on that later), nothing has been written about perhaps the most infamous whiner in magic history. I’m here to fix that. Strap in, because it’s going to get *wild*. Let’s begin. 


**Humble Beginnings**

Though Des’ internet life extends farther than his Magic the Gathering notoriety, that’s a post for another time. He started posting on various message boards under the pseudonym “Desolator114” around 2006, and made his YouTube channel the same year. He remained r

In [99]:
first_res.get('title', '')


'[Magic: The Gathering] Dæsolatormagic; the net deck hating, anime expo shooting, (alleged) card shop owning, most mocked man in Magic'

In [100]:
first_res.get('selftext', '')



'**Background**\n\nFor all the Magic the Gathering posts on this subreddit, it’s shocking to me that one man has not been brought up: DesolatorMagic. Much has been written about his once friend [Jeremy Hambly](https://www.reddit.com/r/HobbyDrama/comments/sw9q7l/magic_the_gathering_the_downfall_of/?utm_source=share&amp;utm_medium=ios_app&amp;utm_name=iossmf) seen [here](https://archive.fo/YL3yz) hanging out together in of an anime convention that Desolator was banned from (more on that later), nothing has been written about perhaps the most infamous whiner in magic history. I’m here to fix that. Strap in, because it’s going to get *wild*. Let’s begin. \n\n\n**Humble Beginnings**\n\nThough Des’ internet life extends farther than his Magic the Gathering notoriety, that’s a post for another time. He started posting on various message boards under the pseudonym “Desolator114” around 2006, and made his YouTube channel the same year. He remained rather quiet for the next ten years, besides fr

## OpenAI query

In [101]:
def call_openai_chat_model(prompt, openai_api_key, max_tokens=3000, temperature=0.5):
    """
    Calls OpenAI's Chat model with the provided prompt.
    """

    response = client.chat.completions.create(model="gpt-4",  # or "gpt-3.5-turbo"
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ],
    max_tokens=max_tokens,
    temperature=temperature)

    return response.choices[0].message.content

### Test

In [105]:
from openai import OpenAI

openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

In [106]:
call_openai_chat_model(prompt=first_res.get('title', ''),
                       openai_api_key=openai_api_key,
                       max_tokens=3000,
                       temperature=0.5)

"Dæsolatormagic is a well-known personality within the Magic: The Gathering community. He is known for his strong stance against net decking, which is the practice of copying successful deck lists from the internet instead of building one's own. He advocates for creativity and originality in deck building, which has earned him both supporters and critics.\n\nHe has also been associated with an incident at an anime expo, although the details of this are not clear. There are rumors that he owns a card shop, but this has not been confirmed.\n\nDespite his controversial opinions and actions, Dæsolatormagic continues to be a significant figure in the Magic: The Gathering community. He is often the subject of mockery, but he also has a dedicated following who appreciate his unique perspective on the game."

## Run together

In [107]:
def truncate_input(input_text, max_tokens=1000):
    """
    Truncates input text to ensure it stays within the token limit (rough approximation).
    """
    words = input_text.split()
    return " ".join(words[:max_tokens])

In [113]:
def query_reddit(
    openai_api_key,
    reddit_client_id,
    reddit_client_secret,
    reddit_user_agent,
    query_text=None,
    file_path=None,
    subreddit=None,
    time_filter=None,
    sort=None,
    limit=10):
    
    """
    This function handles the Reddit search and the OpenAI call.
    """
    # Load the query text
    if file_path and os.path.exists(file_path):
        with open(file_path, 'r') as file:
            input_text = file.read().strip()
    elif query_text:
        input_text = query_text
    else:
        raise ValueError("Either query_text or file_path must be provided.")

    input_text = str(input_text)

    # Truncate input
    input_text = truncate_input(input_text, max_tokens=1000)
    print(f"\n\nInput text for Reddit search: {input_text}\n\n")

    # Perform Reddit search
    reddit_results = reddit_search(
        input_text=input_text,
        client_id=reddit_client_id,
        client_secret=reddit_client_secret,
        user_agent=reddit_user_agent,
        subreddit=subreddit,
        time_filter=time_filter,
        sort=sort,
        limit=limit
    )

    # Process Reddit Results into a summary text
    if not reddit_results:
        return "No Reddit posts found for the query."
    
    print(f"Found {len(reddit_results)} Reddit posts.")

    reddit_summary = ""
    max_summary_length = 3000  # Limit the length of the summary to avoid exceeding token limits
    for idx, post_data in enumerate(reddit_results, 1):
        post_summary = f"{idx}. {post_data.get('title', '')}\n{post_data.get('selftext', '')[:300]}...\n\n"
        if len(reddit_summary) + len(post_summary) > max_summary_length:
            break
        reddit_summary += post_summary

    # Compose prompt
    prompt = (
        f"You are an expert in analyzing Reddit discussions.\n\n"
        f"Here are some posts retrieved based on the query:\n\n{reddit_summary}\n\n"
        f"Given the above posts, answer the following query:\n\n{input_text}\n\n"
    )

    # Truncate the prompt to ensure it stays within token limits
    prompt = truncate_input(prompt, max_tokens=3000)

    # Call OpenAI Chat Model
    response = call_openai_chat_model(prompt, openai_api_key=openai_api_key)

    return response

## Test

In [114]:
INPUT_TEXT = "What is the most popular commander in Magic the Gathering?"

query_reddit(
    openai_api_key=openai_api_key,
    reddit_client_id=CLIENT_ID,
    reddit_client_secret=CLIENT_SECRET,
    reddit_user_agent=USER_AGENT,
    query_text=INPUT_TEXT,
    # subreddit='MagicArena',  # Example subreddit
    time_filter='all',  # Example time filter
    sort='relevance',  # Example sort order
    limit=100  # Limit the number of results
)



Input text for Reddit search: What is the most popular commander in Magic the Gathering?


Found 100 Reddit posts.


"Based on the provided posts, there's no specific information about the most popular commander in Magic the Gathering. The posts mention that Warhammer 40,000 decks were the best selling Commander decks of all time, and that this record has now been surpassed by the Fallout Commander decks. However, they do not specify which individual commander is the most popular. For that information, you may need to look at specific Magic the Gathering community discussions or statistics."